# Notebook 2 of 7: How AI Reads Text

## The Number Problem -- Turning Words Into Something a Computer Can Understand

**Series: Understanding AI Through Italian Music**

---

Here is something that might surprise you: **computers cannot read words.** Not a single one. When you type "Amore mio" on your keyboard, you see two beautiful Italian words meaning "my love." But your computer? It sees nothing but numbers.

This creates a fundamental problem. If we want AI to learn from thousands of Italian song lyrics, we first need to answer a basic question: **how do we convert human language into numbers that a machine can work with?**

The answer is a process called **tokenization** -- and it is the first secret behind how AI works. Every AI system you have ever used -- ChatGPT, Siri, Google Translate -- starts by converting your words into numbers before it can do anything else.

In this notebook, we will:
- See exactly how words become numbers
- Discover that AI does not always split text into whole words (and why that is clever)
- Understand what happens when an English-trained AI encounters Italian
- Learn why all inputs must be the same length

Let's pull back the curtain and see how AI really "reads" text.

## Setup

First, we load the tools we need. The most important one is the **GPT-2 Tokenizer** from the Hugging Face Transformers library. This is the same tokenizer used by GPT-2 -- a predecessor to the models behind ChatGPT.

In [ ]:
# Setup: load the tools we need
import sys
sys.path.insert(0, '..')

from transformers import GPT2Tokenizer
from src.visualization import visualize_tokenization

# Load the GPT-2 tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

print("Tokenizer loaded successfully!")
print(f"This tokenizer knows {tokenizer.vocab_size:,} different tokens.")

---

## What Is a Tokenizer?

Think of a tokenizer as a massive dictionary, but instead of definitions, every entry has a **unique number**:

| Word | Number |
|------|--------|
| "the" | 262 |
| "love" | 7511 |
| "music" | 7849 |
| "is" | 318 |

A **tokenizer** is exactly this kind of dictionary. It takes a piece of text and converts every word (or piece of a word) into its corresponding number. These numbers are called **token IDs**.

Here is the key insight: **the AI never sees the actual words.** It only ever works with these numbers. When you ask ChatGPT a question, your entire message is first converted into a long list of numbers. The AI processes those numbers, produces new numbers as output, and then those output numbers are converted back into words for you to read.

Let's see this in action with a line of Italian.

### Let's See It in Action

We will take the Italian phrase **"Amore mio, ti penso sempre"** (meaning "My love, I think of you always") and watch the tokenizer convert it into numbers.

In [ ]:
# Let's tokenize an Italian phrase
text = "Amore mio, ti penso sempre"

# Convert text to token IDs
token_ids = tokenizer.encode(text)

print(f'Original text:  "{text}"')
print(f"Token IDs:       {token_ids}")
print(f"Number of tokens: {len(token_ids)}")
print()

# Now let's see what each number corresponds to
print("Word-to-Number Mapping:")
print("-" * 40)
print(f"{'Token':<20} {'Token ID':>10}")
print("-" * 40)
for token_id in token_ids:
    token_text = tokenizer.decode([token_id])
    print(f"{repr(token_text):<20} {token_id:>10}")
print("-" * 40)
print()
print('This is what the AI actually "sees" -- just a list of numbers!')

---

## Subword Tokenization: Why AI Does Not Always Use Whole Words

You may have noticed something surprising in the output above: the tokenizer did not always split the text into neat, whole words. Some words were broken into smaller pieces.

This is called **subword tokenization**, and it is one of the cleverest ideas in modern AI.

Here is the problem it solves: if the tokenizer only knew whole words, it would need a separate entry for every word in every language. That would be millions of entries, and it still could not handle a brand new word it had never seen before (like a typo, a name, or a word from an unfamiliar language).

Instead, GPT-2 uses a system called **Byte-Pair Encoding (BPE)**:
- **Common words** stay whole: "the", "and", "love" each get one token
- **Rare or unusual words** get split into smaller pieces that the tokenizer does recognize

Think of it like building with LEGO bricks. You might not have a single brick shaped like "incredibile" (incredible in Italian), but you can build it from smaller pieces: "incred" + "ibile".

This is why AI can handle words it has never seen before -- it just assembles them from familiar building blocks.

Let's see this with some examples.

In [ ]:
# Let's see how the tokenizer handles different words
# Some stay whole, some get split into pieces

examples = [
    ("the", "Common English word"),
    ("love", "Common English word"),
    ("Amore", "Italian for 'love'"),
    ("incredibile", "Italian for 'incredible'"),
    ("meravigliosamente", "Italian for 'wonderfully'"),
    ("spensieratezza", "Italian for 'carefree attitude'"),
]

print("How the tokenizer breaks down different words:")
print("=" * 65)
print(f"{'Word':<22} {'Pieces':<30} {'# Tokens'}")
print("=" * 65)

for word, description in examples:
    ids = tokenizer.encode(word)
    pieces = [tokenizer.decode([t]) for t in ids]
    pieces_str = " | ".join(pieces)
    print(f"{word:<22} {pieces_str:<30} {len(ids)}")

print("=" * 65)
print()
print("Notice how common English words like 'the' and 'love' stay as one token,")
print("while longer Italian words get split into multiple pieces.")
print("This is subword tokenization at work!")

---

## Visualizing Tokens

Numbers and tables are useful, but sometimes a picture is worth a thousand tokens. Let's create a colorful visualization that shows exactly how the tokenizer breaks apart an Italian lyric line.

Each colored block represents one token. Below each block is the numeric ID that the AI actually works with.

In [ ]:
# Visualize how the tokenizer breaks apart an Italian lyric line
# Each colored block = one token, with the token ID shown below

lyric_line = "Nel blu dipinto di blu, felice di stare quassu"
fig = visualize_tokenization(tokenizer, lyric_line)
# ("In the blue painted blue, happy to be up here" -- from the famous song "Volare")

In [ ]:
# Let's also visualize a second lyric line to see a different pattern
lyric_line_2 = "Che bella cosa na jurnata 'e sole"
fig2 = visualize_tokenization(tokenizer, lyric_line_2)
# ("What a beautiful thing, a sunny day" -- from "'O Sole Mio")

---

## The Vocabulary

Every tokenizer has a fixed **vocabulary** -- the complete list of all the tokens it knows. Think of it as the size of its dictionary.

How big is this dictionary? Let's find out, and compare it to human vocabulary.

In [ ]:
# How big is the tokenizer's vocabulary?
print(f"GPT-2 Vocabulary Size: {tokenizer.vocab_size:,} tokens")
print()

# Let's put that in perspective
print("For comparison:")
print(f"  Average adult vocabulary:     ~20,000 - 35,000 words")
print(f"  Shakespeare used:             ~31,000 unique words")
print(f"  GPT-2 tokenizer vocabulary:    {tokenizer.vocab_size:,} word pieces")
print()
print("But remember: GPT-2's vocabulary contains 'word pieces', not just")
print("whole words. It includes common words, word fragments, punctuation,")
print("numbers, and even single characters.")
print()

# Let's peek at some interesting tokens in the vocabulary
print("Some interesting tokens from the vocabulary:")
print("-" * 50)

# Show a sample of tokens across different ID ranges
sample_ids = [0, 1, 10, 50, 262, 318, 7511, 15496, 25, 50256]
for tid in sample_ids:
    token_text = tokenizer.decode([tid])
    print(f"  Token ID {tid:>6}  -->  {repr(token_text)}")

print("-" * 50)
print()
print(f"The very last token (ID {tokenizer.vocab_size - 1}) is a special")
print("'end of text' marker that tells the AI when a piece of text is finished.")

---

## A Critical Insight: Italian with an English Tokenizer

Here is something important to understand about our project: **GPT-2 was trained primarily on English text.** Its vocabulary was built by analyzing millions of English web pages. That means common English words like "the", "and", "love" each get their own single token.

But what about Italian? Since the tokenizer was not designed for Italian, it may not recognize common Italian words as whole units. Instead, it might break them into more pieces -- making Italian text less efficient to process.

This is a real limitation in AI. A model designed for English has to work harder when processing other languages. Let's measure this directly.

In [ ]:
# Let's compare: how many tokens does the same sentence take
# in English vs. Italian?

english_sentence = "The cat is on the table"
italian_sentence = "Il gatto e sul tavolo"

english_tokens = tokenizer.encode(english_sentence)
italian_tokens = tokenizer.encode(italian_sentence)

print("English vs. Italian Token Comparison")
print("=" * 55)
print()

# English
print(f'English: "{english_sentence}"')
print(f"  Tokens: {len(english_tokens)}")
english_pieces = [tokenizer.decode([t]) for t in english_tokens]
print(f"  Breakdown: {english_pieces}")
print()

# Italian
print(f'Italian: "{italian_sentence}"')
print(f"  Tokens: {len(italian_tokens)}")
italian_pieces = [tokenizer.decode([t]) for t in italian_tokens]
print(f"  Breakdown: {italian_pieces}")
print()

print("=" * 55)
diff = len(italian_tokens) - len(english_tokens)
if diff > 0:
    print(f"The Italian version uses {diff} MORE token(s) than English!")
elif diff < 0:
    print(f"The Italian version uses {abs(diff)} FEWER token(s) than English!")
else:
    print("Both versions use the same number of tokens!")
print()
print("This means the AI has to process more numbers to understand the")
print("same meaning in Italian. It is like reading a book where some words")
print("are spelled out letter by letter -- it takes more effort.")
print()

# A more dramatic example with a longer phrase
print()
print("A more dramatic example:")
print("-" * 55)
eng = "I dream of a world full of love and happiness"
ita = "Sogno un mondo pieno di amore e felicita"
eng_tok = tokenizer.encode(eng)
ita_tok = tokenizer.encode(ita)
print(f'English ({len(eng_tok)} tokens): "{eng}"')
print(f'Italian ({len(ita_tok)} tokens): "{ita}"')

---

## Padding and Truncation: Making Everything the Same Size

There is one more thing AI needs before it can work with text: **all inputs must be exactly the same length.**

Why? Think of it like a standardized test form. Every answer sheet has the same number of bubbles, whether you write a one-word answer or a long essay. The machine that reads the forms is built to handle a fixed size.

AI works the same way. We choose a fixed length (for our project, 128 tokens), and then:
- **Short texts get "padded"** -- we fill the remaining space with a special padding token (like leaving bubbles blank)
- **Long texts get "truncated"** -- we cut them off at the maximum length (like running out of space on the form)

Let's see this in action.

In [ ]:
# Padding and truncation example
# We will use a max length of 20 tokens to keep things easy to read

max_length = 20

# A short text
short_text = "Ciao bella"
short_tokens = tokenizer.encode(short_text)

# Set up a pad token (GPT-2 does not have one by default, so we use the
# end-of-text token as our padding token, which is a common practice)
pad_token_id = tokenizer.eos_token_id

# Pad the short text to max_length
padded_tokens = short_tokens + [pad_token_id] * (max_length - len(short_tokens))

print("PADDING EXAMPLE")
print("=" * 60)
print(f'Text: "{short_text}"')
print(f"Original tokens ({len(short_tokens)}):  {short_tokens}")
print(f"Padded to {max_length} tokens:   {padded_tokens}")
print()
print("The real content:")
for i, tid in enumerate(padded_tokens):
    token_text = tokenizer.decode([tid])
    label = "  <-- padding" if i >= len(short_tokens) else ""
    print(f"  Position {i:>2}: {repr(token_text):>12}  (ID: {tid}){label}")

print()
print()

# A long text
long_text = "Nel mezzo del cammin di nostra vita mi ritrovai per una selva oscura che la diritta via era smarrita"
long_tokens = tokenizer.encode(long_text)

# Truncate to max_length
truncated_tokens = long_tokens[:max_length]

print("TRUNCATION EXAMPLE")
print("=" * 60)
print(f'Text: "{long_text}"')
print(f"Original length: {len(long_tokens)} tokens")
print(f"After truncation to {max_length}: {len(truncated_tokens)} tokens")
print()

# Show what was kept and what was lost
kept_text = tokenizer.decode(truncated_tokens)
lost_text = tokenizer.decode(long_tokens[max_length:])
print(f"Kept:  \"{kept_text}\"")
print(f"Lost:  \"{lost_text}\"")
print()
print(f"We lost {len(long_tokens) - max_length} tokens worth of text!")
print("This is why choosing the right max_length matters.")

---

## Key Takeaways

Let's recap what we have learned about how AI reads text:

- **Text becomes numbers through tokenization.** The AI never sees words -- it only works with numeric token IDs. The tokenizer is the translator between human language and machine language.

- **Subword tokenization handles rare words.** Instead of needing a separate entry for every possible word, the tokenizer can build unfamiliar words from smaller, known pieces -- like LEGO bricks.

- **Vocabulary size matters.** GPT-2 knows 50,257 token pieces. A larger vocabulary means fewer splits for common words, but also a bigger model.

- **Using an English tokenizer for Italian is a compromise.** Because GPT-2 was trained on English, Italian words often get split into more tokens than English words. This makes processing less efficient and is a real-world limitation.

- **All inputs must be the same length.** Short texts get padded with empty tokens, and long texts get truncated. This is a requirement of how neural networks process data in batches.

---

## What's Next?

Now that we know how AI reads text -- by converting every word into numbers through tokenization -- we are ready for the exciting part.

In the next notebook, we will **build our first AI model and teach it to write Italian lyrics!** We will start with the simplest type of neural network (an RNN) and watch it learn, word by word, how Italian songs are put together.

**Next: Notebook 3 -- Training a Simple RNN**